# CycleGAN Ergebnis-Viewer (bestehende Outputs)

Wähle ein Experiment unter `results/cyclegan_inference_demo/<experiment>` und zeige die bereits generierten Bilder (day2night/night2day) zusammen mit den Originalen. Es wird **nichts neu berechnet** und **nichts gespeichert**.


In [1]:
from pathlib import Path
import random
import matplotlib.pyplot as plt
from PIL import Image

# --- Parameter ---
experiment_name = 'cyclegan_day2night_20251204_184245'  # Experiment-Ordner unter results/cyclegan_inference_demo
results_root = Path('results/cyclegan_inference_demo') / experiment_name
directions = ['day2night', 'night2day']
samples_per_direction = None  # None = alle Ausgaben zeigen; Zahl = zufällige Auswahl
originals = {
    'day2night': Path('data/bdd_split/cyclegan/testA'),
    'night2day': Path('data/bdd_split/cyclegan/testB'),
}
print('Experiment:', experiment_name)
print('Results root:', results_root)


Experiment: cyclegan_day2night_20251204_184245
Results root: results/cyclegan_inference_demo/cyclegan_day2night_20251204_184245


In [2]:
def direction_root(direction: str) -> Path:
    return results_root / direction


def gather_pairs(direction: str, count: int | None):
    out_dir = direction_root(direction)
    if not out_dir.is_dir():
        print(f'Kein Ordner für {direction}: {out_dir}')
        return []
    orig_root = originals.get(direction)
    if not orig_root or not orig_root.is_dir():
        print(f'Kein Original-Ordner für {direction}: {orig_root}')
        return []
    imgs = [p for p in sorted(out_dir.rglob('*')) if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}]
    if not imgs:
        print(f'Keine Bilder in {out_dir}')
        return []
    if count is not None:
        random.shuffle(imgs)
        imgs = imgs[:count]
    pairs = []
    for path in imgs:
        rel = path.relative_to(out_dir)
        orig = orig_root / rel
        if not orig.is_file():
            continue
        try:
            out_img = Image.open(path).convert('RGB')
            orig_img = Image.open(orig).convert('RGB')
        except Exception as e:
            print(f'Überspringe {path}: {e}')
            continue
        pairs.append((path, orig_img, out_img, orig))
    if not pairs:
        print(f'Keine passenden Originale für {direction}')
    return pairs


def show_pairs(pairs, title: str):
    if not pairs:
        return
    n = len(pairs)
    fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))
    if n == 1:
        axes = [axes]
    for ax_row, (path, orig, out_img, orig_path) in zip(axes, pairs):
        ax_row[0].imshow(orig)
        ax_row[0].set_title(f'Original\n{orig_path.name}')
        ax_row[0].axis('off')
        ax_row[1].imshow(out_img)
        ax_row[1].set_title(f'{title}\n{path.name}')
        ax_row[1].axis('off')
    fig.tight_layout()
    plt.show()


In [3]:
for direction in directions:
    count = None if samples_per_direction is None else int(samples_per_direction)
    pairs = gather_pairs(direction, count)
    show_pairs(pairs, direction.replace('2', ' → '))


Kein Ordner für day2night: results/cyclegan_inference_demo/cyclegan_day2night_20251204_184245/day2night
Kein Ordner für night2day: results/cyclegan_inference_demo/cyclegan_day2night_20251204_184245/night2day
